# 🚗 CoreVision — Car Model Classifier Training

**Model:** EfficientNetV2-S fine-tuned on CompCars dataset  
**Dataset:** CompCars (~1,716 car models, ~137,000 images)  
**Runtime:** T4 GPU (~3-5 hours)

## Instructions
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run all cells top to bottom
3. Final weights will be saved to your **Google Drive** at:
   `MyDrive/CoreVision/weights/car_classifier.pth`
4. Download and place in your project's `weights/` folder

In [ ]:
# ============================================================
# CELL 1: Check GPU & Mount Google Drive
# ============================================================
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CoreVision/weights', exist_ok=True)
os.makedirs('/content/drive/MyDrive/CoreVision/data', exist_ok=True)
print('Drive mounted. Storage ready.')

In [ ]:
# ============================================================
# CELL 2: Install Dependencies
# ============================================================
!pip install -q timm gdown
print('Dependencies installed!')

In [ ]:
# ============================================================
# CELL 3: Download CompCars Dataset
# ============================================================
# CompCars web-nature subset (publicly available)
# Source: http://mmlab.ie.cuhk.edu.hk/datasets/comp_cars/
#
# OPTION A (recommended): Download from Kaggle
# You'll need a Kaggle API key. Go to kaggle.com → Account → API → Create New Token
# Upload the downloaded kaggle.json here:

from google.colab import files
import os

DRIVE_DATA = '/content/drive/MyDrive/CoreVision/data'
LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# Check if dataset already downloaded to Drive
if os.path.exists(f'{DRIVE_DATA}/compcars'):
    print('Dataset found in Drive! Copying to local...')
    !cp -r "{DRIVE_DATA}/compcars" "{LOCAL_DATA}/"
    print('Done!')
else:
    print('Dataset not found in Drive.')
    print('Please upload your kaggle.json to authenticate:')
    # Uncomment and run after uploading kaggle.json:
    # files.upload()  # Upload kaggle.json
    # !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    # !kaggle datasets download -d jessicali9530/stanford-cars-dataset -p {LOCAL_DATA}/
    # !unzip -q {LOCAL_DATA}/stanford-cars-dataset.zip -d {LOCAL_DATA}/cars
    #
    # --- OR manually upload your dataset zip to Drive and run: ---
    # !unzip -q "{DRIVE_DATA}/compcars.zip" -d "{LOCAL_DATA}/"
    print('\n=> After downloading, re-run this cell.')

In [ ]:
# ============================================================
# CELL 4: Dataset Preparation
# ============================================================
# We expect data in ImageFolder format:
#   data/compcars/train/<class_name>/<image>.jpg
#   data/compcars/val/<class_name>/<image>.jpg
#
# If your data is not split yet, this cell creates the split.

import os
import shutil
import random
from pathlib import Path

DATA_ROOT = '/content/data/compcars'

if not os.path.exists(f'{DATA_ROOT}/train'):
    print('Creating train/val split (80/20)...')
    
    # Assume flat structure: DATA_ROOT/<class>/<images>
    source_root = DATA_ROOT  # change if different
    
    train_dir = f'{DATA_ROOT}/train'
    val_dir = f'{DATA_ROOT}/val'
    
    random.seed(42)
    
    for cls in os.listdir(source_root):
        cls_path = os.path.join(source_root, cls)
        if not os.path.isdir(cls_path) or cls in ('train', 'val'):
            continue
        
        images = [f for f in os.listdir(cls_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not images:
            continue
        
        random.shuffle(images)
        split = int(len(images) * 0.8)
        train_imgs = images[:split]
        val_imgs = images[split:]
        
        os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        
        for img in train_imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(train_dir, cls, img))
        for img in val_imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(val_dir, cls, img))
    
    print('Train/val split created!')

# Count classes and images
train_dir = f'{DATA_ROOT}/train'
val_dir = f'{DATA_ROOT}/val'

classes = os.listdir(train_dir)
n_classes = len(classes)
n_train = sum(len(os.listdir(os.path.join(train_dir, c))) for c in classes)
n_val = sum(len(os.listdir(os.path.join(val_dir, c))) for c in os.listdir(val_dir))

print(f'Classes: {n_classes}')
print(f'Train images: {n_train:,}')
print(f'Val images: {n_val:,}')

# Save class names
sorted_classes = sorted(classes)
with open('/content/drive/MyDrive/CoreVision/data/compcars_classes.txt', 'w') as f:
    f.write('\n'.join(sorted_classes))
print(f'Saved {n_classes} class names to Drive.')

In [ ]:
# ============================================================
# CELL 5: Hyperparameters & Configuration
# ============================================================
import os

DATA_ROOT = '/content/data/compcars'

CONFIG = {
    # Data
    'train_dir': f'{DATA_ROOT}/train',
    'val_dir': f'{DATA_ROOT}/val',
    'num_classes': n_classes,
    'img_size': 224,
    
    # Training
    'batch_size': 64,          # Fits on T4 16GB
    'num_epochs': 30,
    'num_workers': 2,
    
    # Optimizer
    'lr_backbone': 1e-4,       # Lower LR for pretrained backbone
    'lr_head': 3e-4,           # Higher LR for new classification head
    'weight_decay': 1e-4,
    
    # Scheduler
    'warmup_epochs': 3,
    
    # Checkpointing
    'save_path': '/content/drive/MyDrive/CoreVision/weights/car_classifier.pth',
    'resume_from': None,       # Set to checkpoint path to resume training
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# CELL 6: Data Loaders with Augmentation
# ============================================================
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = CONFIG['img_size']

# Strong augmentation for training
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),  # Randomly erase patches
])

# No augmentation for validation
val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(CONFIG['train_dir'], transform=train_transforms)
val_dataset = datasets.ImageFolder(CONFIG['val_dir'], transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'],
    pin_memory=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Classes: {len(train_dataset.classes)}')

In [ ]:
# ============================================================
# CELL 7: Build EfficientNetV2-S Model
# ============================================================
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def build_model(num_classes, pretrained=True):
    # EfficientNetV2-S backbone — smaller & faster than V2-M/L, same pattern
    backbone = timm.create_model(
        'tf_efficientnetv2_s',
        pretrained=pretrained,
        num_classes=0  # Remove built-in head
    )
    feature_dim = backbone.num_features  # 1280
    
    model = nn.Sequential(
        backbone,
        nn.Dropout(0.3),
        nn.Linear(feature_dim, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(512, num_classes)
    )
    return model, feature_dim

model, feature_dim = build_model(CONFIG['num_classes'])
model = model.to(device)

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable:,}')
print(f'Feature dim: {feature_dim}')

In [ ]:
# ============================================================
# CELL 8: Optimizer, Scheduler & Loss
# ============================================================
import torch.optim as optim

# Differential learning rates (backbone slower than head)
backbone_params = list(model[0].parameters())
head_params = list(model[1:].parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG['lr_backbone']},
    {'params': head_params, 'lr': CONFIG['lr_head']},
], weight_decay=CONFIG['weight_decay'])

# CosineAnnealingLR: lr smoothly decays to near-zero
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['num_epochs'],
    eta_min=1e-6
)

# Label smoothing: makes model less overconfident, improves generalization
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Mixed precision training (faster on T4)
scaler = torch.cuda.amp.GradScaler()

print('Optimizer:', type(optimizer).__name__)
print('Scheduler:', type(scheduler).__name__)
print('Loss: CrossEntropyLoss with label_smoothing=0.1')
print('AMP: enabled')

In [ ]:
# ============================================================
# CELL 9: Training Loop
# ============================================================
import time

start_epoch = 0
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# Resume from checkpoint
if CONFIG['resume_from'] and os.path.exists(CONFIG['resume_from']):
    ckpt = torch.load(CONFIG['resume_from'], map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt.get('accuracy', 0.0)
    print(f'Resumed from epoch {start_epoch}, best acc: {best_val_acc:.2%}')

def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
        
        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}', end='\r')
    
    return total_loss / total, total_correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, top5_correct, total = 0.0, 0, 0, 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Top-1 accuracy
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        
        # Top-5 accuracy
        _, top5 = outputs.topk(5, dim=1)
        top5_correct += top5.eq(labels.unsqueeze(1)).any(dim=1).sum().item()
        
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    
    return total_loss / total, total_correct / total, top5_correct / total


# ---- Main training loop ----
print(f'Starting training for {CONFIG["num_epochs"]} epochs...')
print('='*60)

for epoch in range(start_epoch, CONFIG['num_epochs']):
    t0 = time.time()
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_acc, val_top5 = val_epoch(model, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    elapsed = time.time() - t0
    
    print(f'Epoch {epoch+1:02d}/{CONFIG["num_epochs"]} '
          f'| Train Loss: {train_loss:.4f} Acc: {train_acc:.2%} '
          f'| Val Loss: {val_loss:.4f} Acc: {val_acc:.2%} Top-5: {val_top5:.2%} '
          f'| {elapsed:.0f}s')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': val_acc,
            'top5_accuracy': val_top5,
            'num_classes': CONFIG['num_classes'],
            'class_names': train_dataset.classes,
        }, CONFIG['save_path'])
        print(f'  ✅ New best! Saved to Drive (val_acc={val_acc:.2%})')

print(f'\nTraining complete! Best val accuracy: {best_val_acc:.2%}')

In [ ]:
# ============================================================
# CELL 10: Plot Training History
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=3)
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss', markersize=3)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=3)
axes[1].plot(epochs, [a*100 for a in history['val_acc']], 'r-o', label='Val Acc', markersize=3)
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.suptitle(f'EfficientNetV2-S on CompCars | Best Val: {best_val_acc:.2%}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CoreVision/training_history.png', dpi=150)
plt.show()
print('Plot saved to Drive!')

In [ ]:
# ============================================================
# CELL 11: Save Class Names & Download Weights
# ============================================================
from google.colab import files

# Save class names locally
with open('/content/compcars_classes.txt', 'w') as f:
    f.write('\n'.join(train_dataset.classes))

print(f'Files in Google Drive at MyDrive/CoreVision/weights/:')
!ls -lh "/content/drive/MyDrive/CoreVision/weights/"
print()
print('Download the following files and put them in your project:')
print('  car_classifier.pth  →  project/weights/car_classifier.pth')
print('  compcars_classes.txt→  project/data/compcars_classes.txt')

# Option: download directly to your PC
# files.download('/content/drive/MyDrive/CoreVision/weights/car_classifier.pth')
# files.download('/content/compcars_classes.txt')